# Erste Schritte 

## Erstes Environement mit Random actions

In [ ]:
import gymnasium as gym
env = gym.make('CartPole-v1', render_mode='human')
obs, info = env.reset()  #obs ist der aktuelle zustand des Wagens und info die zusatzinfo
for i in range(10000):
    action = env.action_space.sample()# macht random aktionen
    obs, reward, terminated, truncated, info = env.step(action)

    if terminated or truncated:
        obs,info = env.reset()
    print(obs)

## Lernen mithilfe der Q gleichung
#### Da wir in dem ersten beispiel unendliche möglichkeiten bei den States haben müssen wir auf ein Environement mit fixen States wechseln

In [ ]:
import gymnasium as gym
import numpy as np
import pygame


alpha= 0.1 #lernrate
gamma=0.99 # disoount


epsilon=1 
epsilon_minimum= 0.01
epsilon_decay= 0.995

q_table = np.zeros([16, 4] )
env = gym.make('FrozenLake-v1', is_slippery=False)
state, info = env.reset()



running = True
for episode in range(20000):
    state, info = env.reset()
    done=False
    while not done:
        
        #aufgabe 3
        if np.random.random()< epsilon:
            action= env.action_space.sample()  
        else:
            action=np.argmax(q_table[state]) 


        new_state, reward, terminated, truncated, info = env.step(action)

        omega= reward + gamma * np.max(q_table[new_state]) - q_table[state, action] #berechnung des TD errors
        q_table[state, action]= q_table[state,action]+alpha * omega 


        state = new_state

        if terminated or truncated:
            done=True
        
    epsilon=max(epsilon_minimum, epsilon * epsilon_decay)
np.save("q_table.npy", q_table)
env.close()


### Testing

In [20]:
import gymnasium as gym
import numpy as np
import pygame

q_table = np.load('q_table.npy')
env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
state, info = env.reset()

running = True
while running:
    action = np.argmax(q_table[state])
    new_state, reward, terminated, truncated, info = env.step(action)
    state=new_state
    if terminated or truncated:
        obs, info = env.reset()

    for event in pygame.event.get():
        if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
            running = False

env.close()






# Jetzt kommt der Cartpole 

In [ ]:
# Preparation
import numpy as np
import gymnasium as gym
import pygame

 #    Position Velocity_Cart  Winkel Winkelgeschwindigkeit
bins  = [6, 8, 25, 15]    #bins festlegen für die Diskretisierung       
lows  = [-2.4, -3.0, -0.21, -3.5]
highs = [ 2.4,  3.0,  0.21,  3.5]

# kanten für die diskretisierung
def kanten(bereichmin,bereichmax, division):
    kanten=np.linspace (bereichmin,bereichmax, division + 1)
    return kanten

# alle kanten in eine liste packen
alle_kanten = [kanten(lows[i], highs[i], bins[i]) for i in range(4)]

# Diskretisierung denn sonst läuft der bereich der winkel gegen unendlich
def discretize(s):
    idx = []
    for i in range(4):
        b = np.digitize(s[i], alle_kanten[i][1:-1])
        b = min(b, bins[i] - 1)
        idx.append(b)
    return tuple(idx)

q_table = np.zeros(bins + [2] )
alpha= 0.1 #lernrate
gamma=0.99 # disoount

#empfohlene werte
epsilon=1 
epsilon_minimum= 0.02
epsilon_decay= 0.997    

episodes=6000

env = gym.make('CartPole-v1')
state, info = env.reset()

running = True
for episode in range(episodes):
    raw_state, info = env.reset()
    state = discretize(raw_state)
    done=False
    while not done:
        if np.random.random()< epsilon:
                    action= env.action_space.sample()  
        else:
            action=np.argmax(q_table[state]) 

        raw_state, reward, terminated, truncated, info = env.step(action)
        new_state = discretize(raw_state)

        done = terminated or truncated

        td= reward + gamma * np.max(q_table[new_state]) - q_table[state +(action,)] #berechnung des TD errors
        q_table[state + (action,)] += alpha * td

        state= new_state

        if terminated or truncated:
                obs, info = env.reset()

    epsilon = max(epsilon_minimum, epsilon * epsilon_decay)
          
np.save("q_table_cartpole.npy", q_table)
env.close()

done = terminated or truncated


## Testing Cartpole 

In [39]:
import gymnasium as gym
import numpy as np
import pygame


bins  = [6, 8, 25, 15]
lows  = [-2.4, -3.0, -0.21, -3.5]
highs = [ 2.4,  3.0,  0.21,  3.5]

def kanten(bereichmin, bereichmax, division):
    return np.linspace(bereichmin, bereichmax, division + 1)

alle_kanten = [kanten(lows[i], highs[i], bins[i]) for i in range(4)]

def discretize(s):
    idx = []
    for i in range(4):
        b = np.digitize(s[i], alle_kanten[i][1:-1])
        b = min(b, bins[i] - 1)
        idx.append(b)
    return tuple(idx)

# --- Tabelle laden, Fenster an ---
q_table = np.load('q_table_cartpole.npy')
env = gym.make('CartPole-v1', render_mode='human')

raw_state, info = env.reset()
state = discretize(raw_state)          

running = True
while running:
    action = np.argmax(q_table[state])                       
    raw_state, reward, terminated, truncated, info = env.step(action)
    state = discretize(raw_state)                            

    if terminated or truncated:
        raw_state, info = env.reset()
        state = discretize(raw_state)                        

    for event in pygame.event.get():
        if event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
            running = False

env.close()